# 🛒 Market Basket Analysis — Association Rule Mining with Apriori

---

| Detail | Info |
|--------|------|
| **Domain** | Retail / E-commerce Analytics |
| **Technique** | Association Rule Mining (Apriori Algorithm) |
| **Library** | mlxtend |
| **Dataset** | 6 customer transactions × 7 unique items |
| **Key Output** | `{jam} → {bread}` with 100% Confidence, 1.2 Lift |
| **Goal** | Discover which products are frequently bought together |

---

## 🎯 Business Objective

> Discover **hidden purchasing patterns** in customer transactions — so a retail store can make smarter decisions about **product placement, cross-selling, and promotions**.

**Real-World Applications:**
- 🛒 **Supermarket shelf layout** — place frequently co-bought items near each other
- 💻 **E-commerce recommendations** — 'Customers who bought X also bought Y'
- 📦 **Bundle promotions** — create discount bundles for co-occurring items
- 📊 **Inventory planning** — stock associated items together

---

## 🧠 What is Association Rule Mining?

Association Rule Mining finds **IF-THEN relationships** in transactional data:

```
IF a customer buys {jam}  →  THEN they also buy {bread}
IF a customer buys {bread} →  THEN they also buy {jam}
```

These rules are evaluated using three key metrics:

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Support** | P(A ∩ B) | How often A and B appear together in ALL transactions |
| **Confidence** | P(B\|A) = P(A∩B)/P(A) | Given A is bought, how often is B also bought? |
| **Lift** | Confidence / P(B) | How much more likely is B bought WITH A vs without A? |

```
Lift > 1  → Positive association (items are bought together more than by chance)
Lift = 1  → No association (independent items)
Lift < 1  → Negative association (items are NOT bought together)
```

---

## 🗺️ Project Workflow

```
Define Transaction Data
          ↓
One-Hot Encoding (TransactionEncoder)
          ↓
Find Frequent Itemsets (Apriori — min_support=0.6)
          ↓
Generate Association Rules (min_confidence=0.6)
          ↓
Filter & Interpret Rules
          ↓
Visualise: Support vs Confidence vs Lift
          ↓
Business Recommendations
```

## 1️⃣ Import Libraries

We need `mlxtend` — a Python library specifically built for frequent pattern mining and association rules.


In [ ]:
# ── Core Libraries ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Association Rule Mining ──
# Install if needed: pip install mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print('✅ All libraries loaded successfully!')


## 2️⃣ Define Transaction Data

### 📊 Dataset Overview

Our dataset contains **6 customer transactions** from a small grocery store.
Each transaction is a list of items purchased together in one shopping trip.

| Transaction | Items Purchased |
|-------------|----------------|
| T1 | milk, bread, rice, book |
| T2 | bread, jam, book, pen |
| T3 | jam, milk, bread, rice, eggs |
| T4 | rice, eggs, pen, book |
| T5 | eggs, pen, milk, bread, jam |
| T6 | eggs, rice, bread, jam |

**Unique items:** book, bread, eggs, jam, milk, pen, rice → **7 items total**

> 📌 **Note:** In real-world scenarios, this dataset could contain millions of transactions from a supermarket POS system. The same algorithm and workflow applies — only the scale changes.


In [ ]:
# Define transaction data — each list = one customer's shopping basket
data = [
    ['milk', 'bread', 'rice', 'book'],        # Transaction 1
    ['bread', 'jam', 'book', 'pen'],           # Transaction 2
    ['jam', 'milk', 'bread', 'rice', 'eggs'], # Transaction 3
    ['rice', 'eggs', 'pen', 'book'],           # Transaction 4
    ['eggs', 'pen', 'milk', 'bread', 'jam'],  # Transaction 5
    ['eggs', 'rice', 'bread', 'jam']           # Transaction 6
]

# Quick overview
all_items = sorted(set(item for transaction in data for item in transaction))
print(f'📊 Total Transactions  : {len(data)}')
print(f'🏷️  Unique Items        : {len(all_items)}')
print(f'📦 Items               : {all_items}')
print(f'📏 Avg Basket Size     : {sum(len(t) for t in data)/len(data):.1f} items')

# Item frequency
from collections import Counter
item_freq = Counter(item for transaction in data for item in transaction)
print(f'\n📊 Item Frequency (count across all transactions):')
for item, count in sorted(item_freq.items(), key=lambda x: -x[1]):
    bar = '█' * count
    pct = count / len(data) * 100
    print(f'  {item:<8} : {bar}  ({count}/6 = {pct:.0f}%)')


In [ ]:
# Visualise item frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
items_sorted = sorted(item_freq.items(), key=lambda x: -x[1])
names, counts = zip(*items_sorted)
support_vals = [c/len(data) for c in counts]

bars = axes[0].bar(names, counts, color='#2E75B6', edgecolor='white', width=0.6)
axes[0].set_title('Item Frequency — How Often Each Item Appears', fontweight='bold')
axes[0].set_ylabel('Count (out of 6 transactions)')
axes[0].set_xlabel('Item')
axes[0].axhline(0.6*len(data), color='red', linestyle='--',
                label=f'min_support threshold (60% = {0.6*len(data):.0f} transactions)')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 str(val), ha='center', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Support values
colors = ['#28a745' if s >= 0.6 else '#dc3545' for s in support_vals]
axes[1].bar(names, support_vals, color=colors, edgecolor='white', width=0.6)
axes[1].axhline(0.6, color='black', linestyle='--', linewidth=2, label='min_support = 0.6')
axes[1].set_title('Item Support — Items Above Threshold in Green', fontweight='bold')
axes[1].set_ylabel('Support (proportion of transactions)')
axes[1].set_xlabel('Item')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[1].patches, support_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()
print('📌 Items with support ≥ 0.6 (green) will be considered for frequent itemsets.')
print('   book, milk, pen fall below threshold → excluded from rules.')


## 3️⃣ One-Hot Encoding — TransactionEncoder

The Apriori algorithm works with a **binary matrix** — it cannot work with raw lists.

We convert each transaction into a row of True/False values:

```
Transaction: ['milk', 'bread', 'rice']

         book   bread   eggs    jam   milk    pen   rice
           False  True   False  False  True   False  True
```

**TransactionEncoder** automatically:
1. Finds all unique items across all transactions
2. Creates a column for each unique item
3. Marks True/False for each transaction row


In [ ]:
# Step 1: Fit TransactionEncoder — learns the full item vocabulary
te = TransactionEncoder()
te_array = te.fit_transform(data)

print(f'✅ One-hot encoding complete!')
print(f'Matrix shape: {te_array.shape}  → ({len(data)} transactions × {len(te.columns_)} items)')
print(f'Item columns: {list(te.columns_)}')

# Step 2: Convert to DataFrame for readability
df = pd.DataFrame(te_array, columns=te.columns_)

# Add transaction labels
df.index = [f'T{i+1}' for i in range(len(data))]
print(f'\n📊 One-Hot Encoded Transaction Matrix:')
df


In [ ]:
# Heatmap of transaction matrix
plt.figure(figsize=(10, 4))
sns.heatmap(df.astype(int), annot=True, cmap='Blues', cbar=False,
            linewidths=0.5, linecolor='gray', fmt='d')
plt.title('Transaction Matrix — 1=Item Present, 0=Absent', fontweight='bold', fontsize=12)
plt.ylabel('Transaction')
plt.xlabel('Item')
plt.tight_layout()
plt.show()
print('📌 Each row = one customer basket. Each column = one product.')
print('   Blue cells = item was purchased in that transaction.')


## 4️⃣ Find Frequent Itemsets — Apriori Algorithm

### How Apriori Works

Apriori uses a **'bottom-up' approach** — starts with individual frequent items, then builds larger itemsets:

```
Step 1: Find all items with support ≥ min_support
  → bread(0.83), eggs(0.67), jam(0.67), rice(0.67)  ✅
  → book(0.50), milk(0.50), pen(0.50)               ❌ (below 0.6)

Step 2: Find all 2-item combinations with support ≥ min_support
  → {jam, bread}(0.67) ✅
  → {eggs, bread}, {rice, jam}... ❌

Step 3: Find all 3-item combinations... (none meet threshold here)
```

### Key Parameter: min_support

| min_support | Effect |
|------------|--------|
| Too high (0.9) | Only very common items — misses interesting patterns |
| Too low (0.1) | Thousands of rules — mostly noise |
| **0.6 (60%)** | Balanced — captures patterns in 3+ of 6 transactions |


In [ ]:
# Generate frequent itemsets
# min_support=0.6 means: item(set) must appear in at least 60% of transactions
itemsets = apriori(df, min_support=0.6, use_colnames=True)

# Add itemset size for analysis
itemsets['itemset_size'] = itemsets['itemsets'].apply(len)
itemsets_sorted = itemsets.sort_values('support', ascending=False)

print(f'✅ Frequent Itemsets Found: {len(itemsets)}')
print(f'   Single items : {len(itemsets[itemsets["itemset_size"]==1])}')
print(f'   Item pairs   : {len(itemsets[itemsets["itemset_size"]==2])}')
print(f'\n📊 All Frequent Itemsets (sorted by support):')
display_df = itemsets_sorted.copy()
display_df['support'] = display_df['support'].round(4)
display_df['support %'] = (display_df['support']*100).round(1).astype(str) + '%'
display_df['itemsets'] = display_df['itemsets'].apply(lambda x: ', '.join(sorted(x)))
print(display_df[['itemsets','support','support %','itemset_size']].to_string(index=False))


In [ ]:
# Visualise frequent itemsets
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Support bar chart
labels = [', '.join(sorted(i)) for i in itemsets_sorted['itemsets']]
supports = itemsets_sorted['support'].values
colors = ['#2E75B6' if s >= 0.7 else '#a8c4e0' for s in supports]

bars = axes[0].barh(labels, supports, color=colors, edgecolor='white')
axes[0].axvline(0.6, color='red', linestyle='--', linewidth=1.5, label='min_support = 0.6')
axes[0].set_title('Frequent Itemsets — Support Values', fontweight='bold')
axes[0].set_xlabel('Support')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, supports):
    axes[0].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                 f'{val:.2f}', va='center', fontweight='bold')

# Itemset size distribution
size_counts = itemsets['itemset_size'].value_counts().sort_index()
axes[1].bar([f'{s}-item set' for s in size_counts.index], size_counts.values,
            color='#2E75B6', edgecolor='white', width=0.4)
axes[1].set_title('Frequent Itemsets by Size', fontweight='bold')
axes[1].set_ylabel('Count')
for bar, val in zip(axes[1].patches, size_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 str(val), ha='center', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 5️⃣ Generate Association Rules

From the frequent itemsets, we now generate **IF-THEN rules** and filter by confidence.

### Key Parameter: min_confidence

| min_confidence | Effect |
|---------------|--------|
| Too high (0.95) | Only near-perfect rules — very few found |
| Too low (0.3) | Weak rules included — lots of noise |
| **0.6 (60%)** | Rules that hold true at least 60% of the time |

### Metrics Explained

| Metric | Rule 1: jam→bread | Rule 2: bread→jam | Interpretation |
|--------|------------------|-------------------|----------------|
| **Support** | 0.667 | 0.667 | 66.7% of all transactions have BOTH items |
| **Confidence** | 1.0 (100%) | 0.8 (80%) | When jam bought → 100% also buy bread |
| **Lift** | 1.2 | 1.2 | 20% MORE likely to buy together than by chance |
| **Leverage** | 0.111 | 0.111 | Positive → co-occurrence above random chance |
| **Conviction** | ∞ | 1.67 | ∞ = perfect rule (jam ALWAYS with bread) |


In [ ]:
# Generate all association rules with min_confidence = 0.6
rules = association_rules(itemsets, metric='confidence', min_threshold=0.6)

print(f'✅ Association Rules Generated: {len(rules)}')
print(f'\n📊 All Rules with Key Metrics:')
rules

In [ ]:
# Clean display — only the most relevant columns
results = rules[['antecedents','consequents','support','confidence','lift']].copy()
results['antecedents'] = results['antecedents'].apply(lambda x: ', '.join(sorted(x)))
results['consequents'] = results['consequents'].apply(lambda x: ', '.join(sorted(x)))
results['rule'] = results['antecedents'] + '  →  ' + results['consequents']
results['support'] = results['support'].round(4)
results['confidence'] = results['confidence'].round(4)
results['lift'] = results['lift'].round(4)

print('📋 Final Association Rules:')
print('─'*60)
for _, row in results.iterrows():
    print(f'  Rule       : {row["rule"]}')
    print(f'  Support    : {row["support"]:.4f}  ({row["support"]*100:.1f}% of transactions)')
    print(f'  Confidence : {row["confidence"]:.4f}  ({row["confidence"]*100:.0f}% reliable)')
    print(f'  Lift       : {row["lift"]:.4f}')
    print('─'*60)


## 6️⃣ Visualise Association Rules

Plotting **Support vs Confidence** with bubble size = Lift — the classic association rule visualisation.

- **X-axis (Support):** How common is the rule?
- **Y-axis (Confidence):** How reliable is the rule?
- **Bubble size (Lift):** How strong is the association?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Support vs Confidence (bubble size = lift)
sc = axes[0].scatter(
    rules['support'], rules['confidence'],
    s=rules['lift']*300, c=rules['lift'],
    cmap='YlOrRd', alpha=0.8, edgecolors='gray', linewidth=1
)
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].set_xlabel('Support', fontsize=11)
axes[0].set_ylabel('Confidence', fontsize=11)
axes[0].set_title('Support vs Confidence\n(Bubble size = Lift)', fontweight='bold')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.1)
axes[0].axhline(0.6, color='blue', linestyle='--', alpha=0.5, label='min_confidence')
axes[0].axvline(0.6, color='red', linestyle='--', alpha=0.5, label='min_support')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Annotate rules
for _, row in rules.iterrows():
    ant = ', '.join(sorted(row['antecedents']))
    con = ', '.join(sorted(row['consequents']))
    axes[0].annotate(f'{ant}→{con}',
                     (row['support'], row['confidence']),
                     textcoords='offset points', xytext=(8,5),
                     fontsize=9, color='#1F4E79', fontweight='bold')

# Plot 2: Rule metrics comparison bar chart
rule_labels = [f'{{jam}}→{{bread}}', f'{{bread}}→{{jam}}']
metrics = {'Support': rules['support'].values,
           'Confidence': rules['confidence'].values,
           'Lift (÷10)': rules['lift'].values/10}

x = np.arange(len(rule_labels))
width = 0.25
colors_m = ['#2E75B6','#28a745','#dc3545']

for i, (metric, vals) in enumerate(metrics.items()):
    bars = axes[1].bar(x + i*width, vals, width, label=metric,
                       color=colors_m[i], edgecolor='white')
    for bar, val in zip(bars, vals):
        actual = val*10 if metric == 'Lift (÷10)' else val
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                     f'{actual:.2f}', ha='center', fontsize=9, fontweight='bold')

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(rule_labels, fontsize=10)
axes[1].set_title('Rule Metrics Comparison', fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].set_ylim(0, 1.3)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 7️⃣ Detailed Rule Interpretation

### 📌 Rule 1: `{jam} → {bread}`

| Metric | Value | Plain English |
|--------|-------|---------------|
| Support | 0.667 (66.7%) | 4 out of 6 transactions contain BOTH jam and bread |
| Confidence | 1.0 (100%) | **Every single customer** who bought jam also bought bread |
| Lift | 1.2 | Buying bread is **20% more likely** when jam is in the cart |
| Conviction | ∞ | **Perfect deterministic rule** — jam buyer ALWAYS buys bread |

### 📌 Rule 2: `{bread} → {jam}`

| Metric | Value | Plain English |
|--------|-------|---------------|
| Support | 0.667 (66.7%) | 4 out of 6 transactions contain BOTH bread and jam |
| Confidence | 0.8 (80%) | 4 out of 5 bread buyers also bought jam |
| Lift | 1.2 | Buying jam is **20% more likely** when bread is in the cart |
| Conviction | 1.67 | Strong but not perfect — 1 bread buyer didn't buy jam |

### 🔑 Key Insights

**1. Strong Bidirectional Association:**
Both jam→bread and bread→jam have lift > 1, confirming a **genuine positive association** — not coincidence.

**2. Asymmetric Relationship:**
```
jam buyer   → 100% buys bread  (perfect rule)
bread buyer → 80% buys jam     (strong but not perfect)
```
This means jam is a **stronger predictor** of bread than vice versa.

**3. bread is the most common item (83.3%):**
Almost everyone buys bread regardless — it's a staple. Jam buyers always add bread.


In [ ]:
# Detailed interpretation printed clearly
print('='*65)
print('🔍 ASSOCIATION RULE ANALYSIS — DETAILED INTERPRETATION')
print('='*65)

for _, row in rules.iterrows():
    ant = list(row['antecedents'])[0]
    con = list(row['consequents'])[0]
    sup = row['support']
    conf = row['confidence']
    lift = row['lift']

    print(f'\n📌 Rule: {{{ant}}} → {{{con}}}')
    print(f'   Support    : {sup:.4f}  → {sup*100:.1f}% of all transactions contain both items')
    print(f'   Confidence : {conf:.4f}  → When {ant} is bought, {conf*100:.0f}% of time {con} is also bought')
    print(f'   Lift       : {lift:.4f}  → Buying {con} is {(lift-1)*100:.0f}% MORE likely when {ant} is present')

    if lift > 1:
        print(f'   ✅ Positive association — items are bought together more than by chance')
    elif lift == 1:
        print(f'   🔵 No association — items are independent')
    else:
        print(f'   ❌ Negative association — items are NOT bought together')

print(f'\n{"="*65}')
print(f'\n📊 SUMMARY:')
print(f'   Most reliable rule : {{jam}} → {{bread}} (100% confidence)')
print(f'   Most common pair   : jam + bread (66.7% support)')
print(f'   Best lift          : 1.2 (both rules) → 20% positive boost')


## 8️⃣ Business Recommendations

Based on the association rules discovered, here are **actionable recommendations** for the store:

---

### 🛒 1. Product Placement Strategy
Place **jam and bread on adjacent shelves or sections** — customers looking for one will easily spot the other.

```
Current layout:  [Dairy] [Bakery] [Condiments]
Recommended  :  [Bakery + Jam] → increases jam sales with bread purchases
```

### 💰 2. Bundle Promotion
Create a **'Breakfast Bundle' promotion**: `Jam + Bread at a discounted price`

Since 100% of jam buyers already buy bread, bundling them:
- Increases average order value
- Incentivises repeat purchases

### 📱 3. E-Commerce Recommendation Engine
```python
# When a customer adds JAM to cart:
if 'jam' in customer_cart:
    recommend('bread')   # 100% confidence
# When a customer adds BREAD to cart:
if 'bread' in customer_cart:
    recommend('jam')     # 80% confidence
```

### 📊 4. Inventory Planning
Since jam and bread are always purchased together, **align restocking cycles** — never let one run out while the other is in stock.

### 📈 5. Scale to Real Data
This same workflow on **real supermarket POS data** (millions of transactions) would reveal:
- Hundreds of product associations
- Seasonal buying patterns
- Regional preference differences


In [ ]:
# Final business summary — clean print output
print('='*60)
print('🏪 MARKET BASKET ANALYSIS — BUSINESS SUMMARY')
print('='*60)
print()
print('📊 Dataset      : 6 transactions, 7 unique items')
print('🔍 Algorithm    : Apriori (min_support=0.6, min_confidence=0.6)')
print('📋 Rules Found  : 2 strong association rules')
print()
print('─'*60)
print('TOP RULES:')
print('─'*60)
print('1. {jam} → {bread}  |  Conf: 100%  |  Lift: 1.2  ⭐ STRONGEST')
print('2. {bread} → {jam}  |  Conf:  80%  |  Lift: 1.2  ✅ STRONG')
print()
print('─'*60)
print('RECOMMENDATIONS:')
print('─'*60)
print('✅ Place jam next to bread on store shelves')
print('✅ Create Breakfast Bundle: Jam + Bread discount')
print('✅ E-commerce: Recommend bread when jam is added to cart')
print('✅ Sync jam & bread restocking schedules')
print('='*60)


## 9️⃣ Explore Different Threshold Settings

What happens when we **lower the thresholds**? More rules are discovered — useful for larger datasets.


In [ ]:
print('📊 Effect of Different min_support Thresholds:')
print('─'*50)
for sup_thresh in [0.3, 0.5, 0.6, 0.7, 0.9]:
    try:
        freq = apriori(df, min_support=sup_thresh, use_colnames=True)
        rules_temp = association_rules(freq, metric='confidence', min_threshold=0.5)
        print(f'  min_support={sup_thresh} → {len(freq):2d} frequent itemsets, {len(rules_temp):2d} rules')
    except:
        print(f'  min_support={sup_thresh} → No rules found')

print()
print('📊 Effect of Different min_confidence Thresholds (at support=0.5):')
print('─'*50)
freq_05 = apriori(df, min_support=0.5, use_colnames=True)
for conf_thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    try:
        rules_temp = association_rules(freq_05, metric='confidence', min_threshold=conf_thresh)
        print(f'  min_confidence={conf_thresh} → {len(rules_temp):2d} rules')
    except:
        print(f'  min_confidence={conf_thresh} → 0 rules')

print()
print('📌 Choosing the right thresholds is a business decision:')
print('   Low thresholds  → More rules, potentially noisy')
print('   High thresholds → Fewer rules, all high-quality')


---

## 📊 Final Project Summary

### ✅ What We Built
An end-to-end **Market Basket Analysis pipeline** that:
1. Defined 6 customer transactions across 7 grocery items
2. Converted raw transaction lists into a binary one-hot encoded matrix
3. Applied **Apriori algorithm** to find frequent itemsets (support ≥ 60%)
4. Generated **association rules** with confidence ≥ 60%
5. Visualised rules using Support vs Confidence scatter plot
6. Interpreted rules with business-ready insights and recommendations
7. Explored threshold sensitivity analysis

### 🧠 Key Concepts Demonstrated

| Concept | Implementation |
|---------|---------------|
| Transaction Encoding | TransactionEncoder → Binary matrix |
| Frequent Itemsets | Apriori with min_support threshold |
| Association Rules | support, confidence, lift, leverage, conviction |
| Rule Interpretation | Directional vs symmetric relationships |
| Business Insights | Product placement, bundling, recommendation engine |
| Threshold Analysis | Impact of varying support and confidence cutoffs |

### 🚀 Future Improvements
- [ ] Apply to real-world dataset (e.g., **Online Retail Dataset** from UCI)
- [ ] Try **FP-Growth algorithm** — faster alternative to Apriori for large datasets
- [ ] Build a **network graph** showing all item relationships visually
- [ ] Add **seasonal analysis** — rules for weekdays vs weekends
- [ ] Deploy as a **product recommendation API**

---

**👩‍💻 Author: Sireesha Ragipati**  
[![GitHub](https://img.shields.io/badge/GitHub-SireeshaRagipati24-black?style=flat&logo=github)](https://github.com/SireeshaRagipati24)  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-blue?style=flat&logo=linkedin)](https://linkedin.com/in/your-linkedin)
